# Machine breakdowns: full demo

A machine with random breakdowns that pre-empt jobs, plus scheduled maintenance, run at two reliability levels.

*Original script: `5 Source Files/Python/segment_3_demo.py` (kept unchanged). Two corrections. (1) A breakdown can interrupt maintenance; the original then crashes at t = 402.64, and here the maintenance step catches it. (2) An interrupted job now queues for the machine again; the original went on working during the repair (erratum C2). With MTBF = 20 h the average completion time is 42.17 h, not 13.04 h. Run the cells in order; the first cell installs SimPy.*

In [ ]:
# Run this cell first.
# In the browser (JupyterLite) it installs the extra packages into this tab;
# in Colab or Anaconda it installs them with pip (already installed = no change).
%pip install simpy

In [2]:
import simpy
import random
import matplotlib.pyplot as plt

**Data collection.**

In [3]:
# Data collection
job_completion_times = []
job_interruptions = []
total_interruptions = 0

**`machine_breakdown_process`** (function): Randomly breaks the machine down.

In [4]:
def machine_breakdown_process(env, machine, mtbf, mttr):
    """
    Randomly breaks the machine down.
    
    Args:
        mtbf: Mean Time Between Failures (hours)
        mttr: Mean Time To Repair (hours)
    """
    global total_interruptions
    
    while True:
        # Work for some time before breaking
        uptime = random.expovariate(1.0 / mtbf)
        yield env.timeout(uptime)
        
        print(f"{env.now:.2f}: ⚠️  MACHINE BREAKDOWN!")
        total_interruptions += 1
        
        # Request machine with highest priority to "seize" it for repairs
        with machine.request(priority=-1, preempt=True) as req:
            yield req
            
            # Repair the machine
            repair_time = random.expovariate(1.0 / mttr)
            yield env.timeout(repair_time)
            
            print(f"{env.now:.2f}: ✓  Machine repaired (took {repair_time:.2f} hours)")

**`job`** (function): A job that uses the machine and can be interrupted.

In [5]:
def job(env, name, machine, processing_time):
    """
    A job that uses the machine and can be interrupted.
    
    Args:
        processing_time: Total time needed to complete the job
    """
    print(f"{env.now:.2f}: {name} arrives (needs {processing_time:.2f} hours)")
    
    arrival_time = env.now
    interruption_count = 0
    remaining_time = processing_time
    
    # Keep trying to complete the job. After a breakdown the job has lost the machine,
    # so it must queue for it again (corrected: the original kept working without the machine).
    while remaining_time > 0:
        with machine.request(priority=1) as req:
            yield req
            print(f"{env.now:.2f}: {name} working (needs {remaining_time:.2f} more hours)")

            try:
                # Try to complete remaining work
                start_time = env.now
                yield env.timeout(remaining_time)

                # Success! Job completed
                completion_time = env.now - arrival_time
                job_completion_times.append(completion_time)
                job_interruptions.append(interruption_count)

                print(f"{env.now:.2f}: {name} COMPLETED (total time: {completion_time:.2f} hours, interruptions: {interruption_count})")
                remaining_time = 0

            except simpy.Interrupt as interrupt:
                # Machine broke down! Record the work done, then queue again
                work_done = env.now - start_time
                remaining_time -= work_done
                interruption_count += 1

                print(f"{env.now:.2f}: {name} INTERRUPTED (completed {work_done:.2f} hours, {remaining_time:.2f} hours remaining)")

**`job_generator`** (function): Generate jobs arriving at the machine.

In [6]:
def job_generator(env, machine, arrival_rate, processing_time_mean):
    """Generate jobs arriving at the machine."""
    
    job_number = 0
    while True:
        # Wait for next job
        yield env.timeout(random.expovariate(arrival_rate))
        
        job_number += 1
        processing_time = random.expovariate(1.0 / processing_time_mean)
        env.process(job(env, f"Job {job_number}", machine, processing_time))

**`preventive_maintenance_process`** (function): Perform scheduled maintenance.

In [7]:
def preventive_maintenance_process(env, machine, maintenance_interval, maintenance_time):
    """Perform scheduled maintenance."""
    
    while True:
        # Wait for maintenance interval
        yield env.timeout(maintenance_interval)
        
        print(f"{env.now:.2f}: Scheduled maintenance starting")
        
        # Request machine for maintenance
        with machine.request(priority=0) as req:  # Higher priority than jobs
            yield req
            try:
                yield env.timeout(maintenance_time)
                print(f"{env.now:.2f}: Maintenance complete")
            except simpy.Interrupt:
                # A breakdown (priority -1, preempt=True) can seize the machine mid-maintenance
                print(f"{env.now:.2f}: Maintenance interrupted by a breakdown")

**`run_simulation`** (function): Run simulation with specified breakdown parameters.

In [8]:
def run_simulation(mtbf, mttr, sim_time=200, arrival_rate=0.1, processing_time=5.0, random_seed=42):
    """Run simulation with specified breakdown parameters."""
    
    # Reset globals
    global job_completion_times, job_interruptions, total_interruptions
    job_completion_times = []
    job_interruptions = []
    total_interruptions = 0
    
    # Setup
    random.seed(random_seed)
    env = simpy.Environment()
    machine = simpy.PreemptiveResource(env, capacity=1)
    
    # Start processes
    env.process(machine_breakdown_process(env, machine, mtbf, mttr))
    env.process(job_generator(env, machine, arrival_rate, processing_time))
    env.process(preventive_maintenance_process(env, machine, maintenance_interval=200, maintenance_time=2))
    
    # Run
    print("="*70)
    print(f"Simulation: MTBF={mtbf}h, MTTR={mttr}h")
    print("="*70)
    env.run(until=sim_time)
    
    # Return statistics
    return {
        'mtbf': mtbf,
        'mttr': mttr,
        'jobs_completed': len(job_completion_times),
        'avg_completion_time': sum(job_completion_times) / len(job_completion_times) if job_completion_times else 0,
        'max_completion_time': max(job_completion_times) if job_completion_times else 0,
        'avg_interruptions': sum(job_interruptions) / len(job_interruptions) if job_interruptions else 0,
        'total_breakdowns': total_interruptions
    }

**Run baseline: No breakdowns (MTBF = very high).**

In [9]:
SIM_TIME = 500
# Run baseline: No breakdowns (MTBF = very high)
print("\n" + "="*70)
print("BASELINE: No Breakdowns")
print("="*70)
baseline = run_simulation(mtbf=10000, mttr=1, sim_time=SIM_TIME)

print(f"\nJobs completed: {baseline['jobs_completed']}")
print(f"Average completion time: {baseline['avg_completion_time']:.2f} hours")
print(f"Maximum completion time: {baseline['max_completion_time']:.2f} hours")


BASELINE: No Breakdowns
Simulation: MTBF=10000h, MTTR=1h
0.25: Job 1 arrives (needs 1.61 hours)
0.25: Job 1 working (needs 1.61 more hours)
1.86: Job 1 COMPLETED (total time: 1.61 hours, interruptions: 0)
2.78: Job 2 arrives (needs 6.67 hours)
2.78: Job 2 working (needs 6.67 more hours)
9.45: Job 2 COMPLETED (total time: 6.67 hours, interruptions: 0)
14.07: Job 3 arrives (needs 11.14 hours)
14.07: Job 3 working (needs 11.14 more hours)
14.98: Job 4 arrives (needs 2.74 hours)
15.28: Job 5 arrives (needs 1.23 hours)
22.32: Job 6 arrives (needs 0.13 hours)
24.54: Job 7 arrives (needs 5.25 hours)
25.21: Job 3 COMPLETED (total time: 11.14 hours, interruptions: 0)
25.21: Job 4 working (needs 2.74 more hours)
27.95: Job 4 COMPLETED (total time: 12.97 hours, interruptions: 0)
27.95: Job 5 working (needs 1.23 more hours)
29.18: Job 5 COMPLETED (total time: 13.90 hours, interruptions: 0)
29.18: Job 6 working (needs 0.13 more hours)
29.32: Job 6 COMPLETED (total time: 6.99 hours, interruptions: 

**Run with moderate breakdowns.**

In [10]:
# Run with moderate breakdowns
print("\n" + "="*70)
print("SCENARIO 1: Moderate Reliability (MTBF=50h)")
print("="*70)
scenario1 = run_simulation(mtbf=50, mttr=5, sim_time=SIM_TIME)

print(f"\nJobs completed: {scenario1['jobs_completed']}")
print(f"Average completion time: {scenario1['avg_completion_time']:.2f} hours")
print(f"Maximum completion time: {scenario1['max_completion_time']:.2f} hours")
print(f"Average interruptions per job: {scenario1['avg_interruptions']:.2f}")
print(f"Total breakdowns: {scenario1['total_breakdowns']}")


SCENARIO 1: Moderate Reliability (MTBF=50h)
Simulation: MTBF=50h, MTTR=5h
0.25: Job 1 arrives (needs 1.61 hours)
0.25: Job 1 working (needs 1.61 more hours)
1.86: Job 1 COMPLETED (total time: 1.61 hours, interruptions: 0)
2.78: Job 2 arrives (needs 6.67 hours)
2.78: Job 2 working (needs 6.67 more hours)
9.45: Job 2 COMPLETED (total time: 6.67 hours, interruptions: 0)
14.07: Job 3 arrives (needs 11.14 hours)
14.07: Job 3 working (needs 11.14 more hours)
14.98: Job 4 arrives (needs 2.74 hours)
15.28: Job 5 arrives (needs 1.23 hours)
22.32: Job 6 arrives (needs 0.13 hours)
24.54: Job 7 arrives (needs 5.25 hours)
25.21: Job 3 COMPLETED (total time: 11.14 hours, interruptions: 0)
25.21: Job 4 working (needs 2.74 more hours)
27.95: Job 4 COMPLETED (total time: 12.97 hours, interruptions: 0)
27.95: Job 5 working (needs 1.23 more hours)
29.18: Job 5 COMPLETED (total time: 13.90 hours, interruptions: 0)
29.18: Job 6 working (needs 0.13 more hours)
29.32: Job 6 COMPLETED (total time: 6.99 hours

**Run with frequent breakdowns.**

In [11]:
# Run with frequent breakdowns
print("\n" + "="*70)
print("SCENARIO 2: Poor Reliability (MTBF=20h)")
print("="*70)
scenario2 = run_simulation(mtbf=20, mttr=5, sim_time=SIM_TIME)

print(f"\nJobs completed: {scenario2['jobs_completed']}")
print(f"Average completion time: {scenario2['avg_completion_time']:.2f} hours")
print(f"Maximum completion time: {scenario2['max_completion_time']:.2f} hours")
print(f"Average interruptions per job: {scenario2['avg_interruptions']:.2f}")
print(f"Total breakdowns: {scenario2['total_breakdowns']}")


SCENARIO 2: Poor Reliability (MTBF=20h)
Simulation: MTBF=20h, MTTR=5h
0.25: Job 1 arrives (needs 1.61 hours)
0.25: Job 1 working (needs 1.61 more hours)
1.86: Job 1 COMPLETED (total time: 1.61 hours, interruptions: 0)
2.78: Job 2 arrives (needs 6.67 hours)
2.78: Job 2 working (needs 6.67 more hours)
9.45: Job 2 COMPLETED (total time: 6.67 hours, interruptions: 0)
14.07: Job 3 arrives (needs 11.14 hours)
14.07: Job 3 working (needs 11.14 more hours)
14.98: Job 4 arrives (needs 2.74 hours)
15.28: Job 5 arrives (needs 1.23 hours)
20.40: ⚠️  MACHINE BREAKDOWN!
20.40: Job 3 INTERRUPTED (completed 6.33 hours, 4.81 hours remaining)
20.54: ✓  Machine repaired (took 0.13 hours)
20.54: Job 4 working (needs 2.74 more hours)
22.32: Job 6 arrives (needs 5.25 hours)
23.28: Job 4 COMPLETED (total time: 8.30 hours, interruptions: 0)
23.28: Job 5 working (needs 1.23 more hours)
24.51: Job 5 COMPLETED (total time: 9.23 hours, interruptions: 0)
24.51: Job 3 working (needs 4.81 more hours)
24.97: ⚠️  MAC